# Step 3 — Gold Layer: Time Series Analytics

Gold is built from Silver. Its job is to answer questions **across time**, not just at a single snapshot.

Silver asks: *What was BTC's price at 14:17 UTC?*  
Gold asks: *How has BTC's price moved across all our snapshots? How volatile is it? Is dominance rising or falling?*

```
Silver (clean snapshots, one row per coin per timestamp)
        │
        │  sort by coin + time → compute across-time metrics
        ▼
Gold (one row per coin per timestamp, with rolling/derived columns)
```

**Note:** With only a few snapshots, rolling averages aren't statistically meaningful yet.  
Run the scheduler for a day or two, then re-run this notebook — the charts will tell a real story.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

project_root = Path('../../').resolve()
silver_path = project_root / '01-data' / 'silver' / 'crypto_clean.csv'
gold_path   = project_root / '01-data' / 'gold'   / 'crypto_gold.csv'

print('Libraries loaded OK')

## Step 1 — Load Silver and sort by coin + time

The key thing here: we sort by `(symbol, collected_at)`.  
Every calculation we do next (rolling averages, price deltas) depends on the rows being in the right order per coin.

In [ ]:
df = pd.read_csv(silver_path)
df['collected_at'] = pd.to_datetime(df['collected_at'], utc=True)

# Sort: group by coin, then chronological within each coin
df = df.sort_values(['symbol', 'collected_at']).reset_index(drop=True)

snapshots = df['collected_at'].nunique()
coins     = df['symbol'].nunique()
span_mins = (df['collected_at'].max() - df['collected_at'].min()).seconds // 60

print(f'Coins:     {coins}')
print(f'Snapshots: {snapshots}')
print(f'Time span: {span_mins} minutes')
print()
print('Snapshot times:')
for t in sorted(df['collected_at'].unique()):
    print(f'  {t}')

## Step 2 — Observed price delta between snapshots

The API gives us `percent_change_24h` — that's the change over the **last 24 hours from the API's perspective**, not between our collection snapshots.

We want to know: **how much did price actually change between our snapshot at time T and our snapshot at time T-1?**

`groupby('symbol')['price'].pct_change()` does exactly this — for each coin, it computes the % change from the previous row (which is the previous snapshot, since we sorted by time).

The first snapshot for each coin will be NaN — there's no previous snapshot to compare to.

In [ ]:
# Price change between consecutive snapshots (our own observed delta)
df['price_delta_pct'] = (
    df.groupby('symbol')['quote_USD_price']
    .pct_change() * 100
)

# Time gap between snapshots (in minutes) — useful for normalizing deltas
df['minutes_since_last'] = (
    df.groupby('symbol')['collected_at']
    .diff()
    .dt.total_seconds() / 60
)

# Log return of observed delta
df['observed_log_return'] = np.log(1 + df['price_delta_pct'] / 100)

# Show latest snapshot's observed deltas
latest = df[df['collected_at'] == df['collected_at'].max()]

print('Price delta from previous snapshot (most recent):')
print(
    latest[['symbol', 'quote_USD_price', 'price_delta_pct', 'minutes_since_last']]
    .sort_values('price_delta_pct', ascending=False)
    .to_string(index=False, float_format='{:+.4f}'.format)
)

## Step 3 — Rolling price averages (Simple Moving Average)

A **Simple Moving Average (SMA)** smooths out noise by averaging the last N snapshots.

`window=3` means: average of last 3 snapshots.  
`min_periods=1` means: compute even if fewer than 3 snapshots exist (so early rows don't become NaN).

When you have more data, increase the window. With 4 snapshots, a window of 3 is the maximum meaningful size.

In [ ]:
df['price_sma_3']  = df.groupby('symbol')['quote_USD_price'].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean()
)

df['price_sma_7']  = df.groupby('symbol')['quote_USD_price'].transform(
    lambda x: x.rolling(window=7, min_periods=1).mean()
)

# Show BTC's price vs moving averages across all snapshots
btc = df[df['symbol'] == 'BTC'][['collected_at', 'quote_USD_price', 'price_sma_3', 'price_sma_7']]
print('BTC — price vs moving averages:')
print(btc.to_string(index=False, float_format='{:,.2f}'.format))

## Step 4 — Rolling volatility

**Realized volatility** = rolling standard deviation of log returns.

High volatility = price is jumping around a lot.  
Low volatility = price is stable.

With only 4 snapshots this will be noisy, but the structure is right.  
Once you have 30+ snapshots, this becomes a real signal.

In [ ]:
df['volatility_3']  = df.groupby('symbol')['observed_log_return'].transform(
    lambda x: x.rolling(window=3, min_periods=2).std()
)

# Show volatility ranking at most recent snapshot
latest = df[df['collected_at'] == df['collected_at'].max()].copy()
latest = latest[~latest['is_stablecoin']]  # exclude stablecoins

print('Volatility at latest snapshot (excluding stablecoins):')
print(
    latest[['symbol', 'quote_USD_price', 'volatility_3']]
    .sort_values('volatility_3', ascending=False)
    .to_string(index=False, float_format='{:.6f}'.format)
)

## Step 5 — Market cap dominance shift

BTC dominance (% of total crypto market cap held by BTC) is one of the most watched macro signals in crypto.

Rising dominance → money flowing into BTC (usually risk-off sentiment).  
Falling dominance → money flowing into altcoins (usually risk-on, called 'altseason').

Here we track how dominance has shifted across our snapshots.

In [ ]:
df['dominance_delta'] = (
    df.groupby('symbol')['quote_USD_market_cap_dominance']
    .diff()
)

# Show dominance change for each coin
latest = df[df['collected_at'] == df['collected_at'].max()]

print('Market cap dominance at latest snapshot:')
print(
    latest[['symbol', 'quote_USD_market_cap_dominance', 'dominance_delta']]
    .sort_values('quote_USD_market_cap_dominance', ascending=False)
    .to_string(index=False, float_format='{:+.4f}'.format)
)

## Step 6 — Visualise: price over time per coin

With only a few snapshots, this chart won't look like much.  
Re-run this notebook after a day or two of the scheduler running — you'll see real price movement.

In [ ]:
# Exclude stablecoins — their flat $1 line will squash everything else
plot_df = df[~df['is_stablecoin']]

fig, ax = plt.subplots(figsize=(12, 5))

for symbol, group in plot_df.groupby('symbol'):
    # Normalise to 100 at first snapshot so all coins are on the same scale
    first_price = group['quote_USD_price'].iloc[0]
    normalised  = group['quote_USD_price'] / first_price * 100
    ax.plot(group['collected_at'], normalised, marker='o', label=symbol)

ax.axhline(100, color='gray', linestyle='--', linewidth=0.8, label='baseline (100)')
ax.set_title('Normalised price (base = 100 at first snapshot)')
ax.set_xlabel('Time (UTC)')
ax.set_ylabel('Normalised price')
ax.legend(loc='upper left', fontsize=8)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## Step 7 — Save Gold

Unlike Bronze and Silver, Gold is **overwritten** (not appended).  
Why? Because Gold is derived from all of Silver — every time you rebuild it, you recompute all rolling metrics from scratch using all available history.

Appending Gold would cause duplicates. Overwriting it is correct.

In [ ]:
gold_path.parent.mkdir(parents=True, exist_ok=True)

# Overwrite — Gold is always rebuilt from all Silver data
df.to_csv(gold_path, index=False)

print(f'Saved to: {gold_path}')
print(f'Rows:     {len(df)}')
print(f'Columns:  {df.shape[1]}')

print()
print('New Gold columns (vs Silver):')
silver_cols = set(pd.read_csv(silver_path).columns)
gold_cols   = set(df.columns)
for col in sorted(gold_cols - silver_cols):
    print(f'  + {col}')

---
## What you just built

```
Silver (clean snapshots)
    ↓  sort by (coin, time)
    ↓  price_delta_pct     ← actual observed change between OUR snapshots
    ↓  price_sma_3/7       ← smoothed price
    ↓  volatility_3        ← rolling std of log returns
    ↓  dominance_delta     ← how market cap share is shifting
Gold (time-series ready, overwritten each run)
```

**Key insight — why Bronze appends but Gold overwrites:**
- Bronze: each API call adds new raw data. Never recompute, never overwrite.
- Silver: each run adds new cleaned rows from that run's Bronze data.
- Gold: built from ALL of Silver every time. Recomputing rolling metrics from a partial view would give wrong results.

**Next steps once you have more data:**
1. Come back and re-run this notebook — the chart in Step 6 will show real price movement
2. Add more indicators: RSI, Bollinger Bands, MACD (we'll build those in the analysis phase)
3. Start the EDA notebooks — understanding coin behavior patterns